# 📚 Knowledge Base & Data Tools
**Run this notebook once** to build the vector knowledge base from 163 PDF daily drilling reports and validate sensor data access.

Outputs:
- `chroma_db/` — Persisted ChromaDB vector store (used by Notebook 02)
- Validates CSV sensor data loading

## 1. Install Dependencies

In [ ]:
!pip install -q langchain langchain-openai langchain-community langgraph chromadb sentence-transformers python-dotenv pymupdf pandas numpy scikit-learn openai

## 2. Configuration

In [ ]:
import os, re, hashlib, warnings
from datetime import datetime
import numpy as np
import pandas as pd

# Paths
REPORT_DIR = os.path.join(os.getcwd(), "16A(78)-32_Daily_Reports", "drilling")
if not os.path.isdir(REPORT_DIR):
    REPORT_DIR = os.path.join(os.getcwd(), "16A(78)-32_Daily_Reports", "16A(78)-32_Daily_Reports", "drilling")
CSV_PATH = os.path.join(os.getcwd(), "16A(78)-32_time_data_10s_intervals.csv")
CHROMA_DIR = os.path.join(os.getcwd(), "chroma_db")

print(f"Report dir exists: {os.path.isdir(REPORT_DIR)}")
print(f"CSV exists: {os.path.isfile(CSV_PATH)}")
print(f"ChromaDB dir: {CHROMA_DIR}")

## 3. Build Knowledge Base from PDFs

In [ ]:
import pymupdf
import chromadb
from chromadb.config import Settings
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# Event patterns for tagging chunks
EVENT_PATTERNS = [
    (re.compile(r"short\s*trip", re.I), "short_trip"),
    (re.compile(r"wiper\s*trip", re.I), "wiper_trip"),
    (re.compile(r"back\s*ream", re.I), "back_ream"),
    (re.compile(r"ream(?:ing)?\s+(?:from|curve)", re.I), "reaming"),
    (re.compile(r"tight\s*spot", re.I), "tight_spot"),
    (re.compile(r"high\s+torque", re.I), "high_torque"),
    (re.compile(r"pull\s+out\s+of\s+hole|POOH", re.I), "trip_out"),
    (re.compile(r"stuck\s+pipe", re.I), "stuck_pipe"),
    (re.compile(r"pack\s*off", re.I), "pack_off"),
    (re.compile(r"over\s*pull", re.I), "overpull"),
    (re.compile(r"drag", re.I), "drag"),
    (re.compile(r"wash\s+down|wash\s+up", re.I), "wash"),
]

def extract_report(filepath):
    """Extract text and metadata from a PDF report."""
    doc = pymupdf.open(filepath)
    text = "".join(page.get_text() for page in doc)
    doc.close()
    meta = {"source_file": os.path.basename(filepath)}
    m = re.search(r"RPT\s*DATE:\s*(\d{1,2}/\d{1,2}/\d{4})", text)
    if m:
        try:
            d = datetime.strptime(m.group(1), "%m/%d/%Y")
            meta["date"] = d.strftime("%Y-%m-%d")
            meta["date_display"] = d.strftime("%B %d, %Y")
        except: pass
    m = re.search(r"MD/TVD:\s*([\d,]+)", text)
    if m:
        try: meta["depth_md"] = float(m.group(1).replace(",", ""))
        except: pass
    events = {et for p, et in EVENT_PATTERNS if p.search(text)}
    if events: meta["event_types"] = ",".join(sorted(events))
    return text, meta

def chunk_text(text, chunk_size=500, overlap=100):
    """Split text into overlapping word-based chunks."""
    words = text.split()
    wpc = int(chunk_size * 0.75)
    olap = int(overlap * 0.75)
    chunks, start = [], 0
    while start < len(words):
        c = " ".join(words[start:start+wpc])
        c = re.sub(r"\s+", " ", c).strip()
        if len(c) > 50: chunks.append(c)
        start += wpc - olap
    return chunks

print("Functions defined.")

In [ ]:
# Build the vector store
embedding_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path=CHROMA_DIR, settings=Settings(anonymized_telemetry=False))

# Delete existing collection if rebuilding
try: client.delete_collection("daily_reports")
except: pass

collection = client.get_or_create_collection("daily_reports", embedding_function=embedding_fn)

pdf_files = sorted(f for f in os.listdir(REPORT_DIR) if f.lower().endswith(".pdf"))
print(f"Found {len(pdf_files)} PDF reports")

all_chunks, all_ids, all_metas = [], [], []
seen = set()

for i, fname in enumerate(pdf_files):
    text, meta = extract_report(os.path.join(REPORT_DIR, fname))
    if not text.strip(): continue
    for j, chunk in enumerate(chunk_text(text)):
        cid = hashlib.md5(f"{fname}:{j}:{chunk[:100]}".encode()).hexdigest()
        if cid in seen: continue
        seen.add(cid)
        cmeta = {"source_file": fname, "chunk_index": j}
        for k in ["date", "date_display", "depth_md"]: 
            if k in meta: cmeta[k] = meta[k]
        chunk_events = {et for p, et in EVENT_PATTERNS if p.search(chunk)}
        if chunk_events: cmeta["event_types"] = ",".join(sorted(chunk_events))
        all_chunks.append(chunk); all_ids.append(cid); all_metas.append(cmeta)
    if (i+1) % 40 == 0: print(f"  Processed {i+1}/{len(pdf_files)}...")

# Add to ChromaDB in batches
for s in range(0, len(all_chunks), 100):
    e = min(s+100, len(all_chunks))
    collection.add(documents=all_chunks[s:e], ids=all_ids[s:e], metadatas=all_metas[s:e])

print(f"\n✅ Knowledge base built: {len(pdf_files)} reports → {len(all_chunks)} chunks")
print(f"   Stored in: {CHROMA_DIR}")

## 4. Test Knowledge Base Queries

In [ ]:
results = collection.query(query_texts=["wiper trip due to high torque"], n_results=3)

for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"[{i+1}] Date: {meta.get('date', 'N/A')} | Events: {meta.get('event_types', 'none')}")
    print(f"    {doc[:200]}...\n")

In [ ]:
results = collection.query(query_texts=["stuck pipe or pack off incident"], n_results=3)

for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"[{i+1}] Date: {meta.get('date', 'N/A')} | Depth: {meta.get('depth_md', 'N/A')}")
    print(f"    {doc[:200]}...\n")

## 5. Load & Validate Sensor Data

In [ ]:
COL_MAP = {
    "Time": "Time", "Weight on Bit": "WOB", "ROP Depth/Hour": "ROP",
    "Top Drive RPM": "RPM", "Top Drive Torque (ft-lbs)": "TRQ",
    "Flow In": "FLOW_IN", "Pump Pressure": "SPP", "SPM Total": "SPM",
    "Pit Volume Active": "PIT_VOL", "Bit RPM": "BIT_RPM",
    "Depth Hole TVD": "DEPTH", "Differential Pressure": "DIFF_P",
    "Downhole Torque": "DH_TRQ", "MUD TEMP": "MUD_TEMP",
    "Block Position": "BLOCK_POS", "Hookload": "HOOKLOAD",
    "Slips Set": "SLIPS", "On Bottom": "ON_BOTTOM",
    "Gas Total - units": "GAS", "Return Flow": "RETURN_FLOW",
    "Pit G/L Active": "PIT_GL", "Trip Volume Active": "TRIP_VOL",
    "Trip G/L": "TRIP_GL", "Total Depth": "TOTAL_DEPTH",
    "RigEventCode": "RIG_EVENT", "Drill Mode": "DRILL_MODE",
    "MWD Inclination": "MWD_INC",
}

df = pd.read_csv(CSV_PATH)
df.rename(columns=COL_MAP, inplace=True)
df = df.iloc[::10].reset_index(drop=True)  # subsample
df["Time"] = pd.to_datetime(df["Time"], format="mixed", dayfirst=False)

# Clean sentinels
num_cols = df.select_dtypes(include="number").columns
df[num_cols] = df[num_cols].replace(-999.25, np.nan)

sensor_cols = [c for c in ["WOB","ROP","RPM","TRQ","FLOW_IN","SPP","HOOKLOAD",
    "DH_TRQ","DIFF_P","BLOCK_POS","GAS","RETURN_FLOW","PIT_GL",
    "TRIP_VOL","TRIP_GL","TOTAL_DEPTH","MWD_INC","MUD_TEMP"] if c in df.columns]
df[sensor_cols] = df[sensor_cols].ffill().fillna(0)

print(f"Sensor data loaded: {len(df):,} rows x {len(df.columns)} cols")
print(f"Time range: {df['Time'].min()} → {df['Time'].max()}")
print(f"Depth range: {df['DEPTH'].min():.0f} → {df['DEPTH'].max():.0f} m")
df.head()

In [ ]:
KEY_PARAMS = ["WOB", "ROP", "RPM", "TRQ", "SPP", "FLOW_IN", "HOOKLOAD", "DH_TRQ", "DIFF_P"]
df[KEY_PARAMS].describe().round(2)

## ✅ Done
Knowledge base is built and persisted to `chroma_db/`. Sensor data validated. Proceed to **`02_agentic_advisor.ipynb`**.